<a href="https://colab.research.google.com/github/felipeafigueiredo/advanced_rag/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Baseline RAG — Projeto IA e AM (F2: How to Search)
Execute as células em ordem. Antes: salve `GEMINI_API_KEY` em *Secrets* e ajuste `REPO_URL` na célula 2.

In [ ]:
# [célula 1] Instalação
!pip -q install google-genai pymupdf pandas numpy

In [ ]:
# [célula 2] Configuração (ajuste só a seção "AJUSTE")
import os, re, json, time, glob, unicodedata, subprocess, getpass, hashlib, shutil
import numpy as np
import pandas as pd
import pymupdf   # PyMuPDF
from google import genai
from google.genai import types

# ----------------------------- AJUSTE -----------------------------------

REPO_URL = "https://github.com/felipeafigueiredo.git"  # repo público com corpus/ e questions.json
REPO_DIR = "repo"          # pasta local onde o repo será clonado
N_RUNS = 3                 # rodadas do baseline (r1, r2, r3)
FORCE_RERUN = False        # True = refaz rodadas cujo CSV já existe

# ------------------------------------------------------------------------

# Modelos (Gemini API)
GEN_MODEL = "gemini-2.5-flash"
EMB_MODEL = "gemini-embedding-001"

# Hiperparâmetros do baseline (mantenha FIXOS: as técnicas da F2 serão comparadas contra estes valores)
CHUNK_SIZE = 1000          # caracteres por chunk
CHUNK_OVERLAP = 200        # sobreposição em caracteres
TOP_K = 5                  # chunks recuperados por pergunta
TEMPERATURE = 0.2
MAX_OUTPUT_TOKENS = 1024

# Limites do plano gratuito (ajuste se usar plano pago: MIN_INTERVAL_GEN = 0)
MIN_INTERVAL_GEN = 6.5     # segundos entre chamadas de geração
EMB_BATCH = 50             # textos por requisição de embedding
EMB_SLEEP = 1.0            # pausa entre lotes de embedding

# Preços em USD por 1M de tokens — CUSTO NOCIONAL (no free tier o custo real é 0).
PRICE_IN_PER_M = 0.30      # entrada do gemini-2.5-flash
PRICE_OUT_PER_M = 2.50     # saída (inclui tokens de raciocínio)
PRICE_EMB_PER_M = 0.15     # gemini-embedding-001

# Chave da API: Colab -> Secrets (ícone da chave) -> nome GEMINI_API_KEY (acesso ao notebook ligado)
try:
    from google.colab import userdata
    API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    API_KEY = os.environ.get("GEMINI_API_KEY") or getpass.getpass("Cole sua GEMINI_API_KEY: ")

client = genai.Client(api_key=API_KEY)

# Repositório / caminhos
if not os.path.isdir(REPO_DIR):
    if "SEU_USUARIO" in REPO_URL:
        raise ValueError("Ajuste REPO_URL na célula 2 (ou envie as pastas corpus/ e questions.json para ./repo).")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

CORPUS_DIR = f"{REPO_DIR}/corpus"
QUESTIONS_PATH = f"{REPO_DIR}/questions.json"
RESULTS_DIR = f"{REPO_DIR}/results"
INDEX_DIR = "index_cache"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(INDEX_DIR, exist_ok=True)
print("PDFs em corpus/:", len(glob.glob(f"{CORPUS_DIR}/*.pdf")))

In [ ]:
# [célula 3] Utilitários: retry, tempo e custo
def call_with_retry(fn, max_attempts=8):
    """Executa fn() com retry em erros transitórios (429/5xx).
    Retorna (resultado, latência_da_tentativa_bem_sucedida_em_s, nº_de_retries).
    O tempo de espera entre tentativas NÃO entra na latência medida."""
    for attempt in range(max_attempts):
        t0 = time.perf_counter()
        try:
            res = fn()
            return res, time.perf_counter() - t0, attempt
        except Exception as e:
            msg = str(e)
            transient = any(k in msg for k in ("429", "RESOURCE_EXHAUSTED", "500", "503", "UNAVAILABLE", "INTERNAL", "DEADLINE"))
            if not transient or attempt == max_attempts - 1:
                raise
            wait = min(15 * (attempt + 1), 90)
            print(f"  [retry {attempt + 1}/{max_attempts - 1}] erro transitório; aguardando {wait}s")
            time.sleep(wait)


def est_tokens(text):
    """Estimativa grosseira (~4 caracteres/token) — a API de embedding não devolve contagem de tokens."""
    return max(1, len(text) // 4)


def cost_usd(tok_in=0, tok_out=0, tok_emb=0):
    return tok_in / 1e6 * PRICE_IN_PER_M + tok_out / 1e6 * PRICE_OUT_PER_M + tok_emb / 1e6 * PRICE_EMB_PER_M


_last_gen_end = [0.0]
def throttle_generation():
    wait = MIN_INTERVAL_GEN - (time.time() - _last_gen_end[0])
    if wait > 0:
        time.sleep(wait)

In [ ]:
# [célula 4] Carregar PDFs, limpar texto e identificar cada documento
DOC_TITLES = {
    "regulamento_graduacao": "Resolução 016/2023-CONSEPE — Regulamento dos Cursos de Graduação",
    "res_013_2022": "Resolução 013/2022-CONSEPE — Reocupação de Vagas Residuais",
    "in_05_2022": "IN 5/2022-PROGRAD — Colação de Grau Individual",
    "in_08_2023": "IN 8/2023-PROGRAD — Depósito de Trabalho de Conclusão de Curso",
    "in_10_2025": "IN 10/2025-PROGRAD — Registro de Diplomas Digitais (IES do RN)",
    "in_11_2026": "IN 11/2026-PROGRAD — Revalidação de Diploma Estrangeiro (REVALIDA)",
}

# Padrões aplicados ao nome do arquivo normalizado (sem acento/pontuação, minúsculo).
# Funcionam com o nome original, com nomes mangled (ex.: "Reocupa__o") ou com o próprio id.
DOC_PATTERNS = {
    "regulamento_graduacao": r"regulamento dos cursos|regulamento graduacao",
    "res_013_2022": r"reocupa|res 013 2022",
    "in_05_2022": r"grau individual|in 05 2022",
    "in_08_2023": r"dep\s?sito|in 08 2023",
    "in_10_2025": r"diplomas digitais|in 10 2025",
    "in_11_2026": r"revalida|in 11 2026",
}


def norm_name(s):
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode().lower()
    return re.sub(r"[^a-z0-9]+", " ", s).strip()


NOISE_LINES = [
    r"^\s*Firefox\s*(https?://\S+.*)?$",              # rodapé de impressão do navegador (SIPAC)
    r"^\s*https?://\S*sipac\S*\s*$",                   # URL do SIPAC repetida no rodapé
    r"^\s*\d+\s+of\s+\d+(\s+\d{2}/\d{2}/\d{4}.*)?$",  # "3 of 7" / "3 of 7 19/04/2023, 17:22"
    r"^\s*\d{2}/\d{2}/\d{4},?\s+\d{2}:\d{2}\s*$",      # "19/04/2023, 17:22"
    r"^\s*\d+\s*/\s*\d+\s*$",                          # "3/7"
]
NOISE_TAIL = r"Visualize o documento original em https?://sipac.*$"


def clean_text(raw):
    lines = []
    for ln in raw.replace("\r", "\n").split("\n"):
        if any(re.match(p, ln) for p in NOISE_LINES):
            continue
        lines.append(ln)
    text = " ".join(lines)
    text = re.sub(NOISE_TAIL, "", text, flags=re.S)
    text = text.replace("­", "")                 # hífen suave
    text = text.encode("utf-8", "ignore").decode("utf-8")   # remove surrogates inválidos que alguns PDFs geram
    text = re.sub(r"\s+", " ", text).strip()
    return text


def load_documents():
    files = sorted(glob.glob(f"{CORPUS_DIR}/*.pdf"))
    if not files:
        raise FileNotFoundError(f"Nenhum PDF em {CORPUS_DIR}/")
    docs = {}
    for doc_id, pat in DOC_PATTERNS.items():
        hits = [f for f in files if re.search(pat, norm_name(os.path.basename(f)))]
        if len(hits) != 1:
            raise ValueError(f"Documento '{doc_id}': esperava 1 arquivo, achei {len(hits)} -> {hits}\n"
                             f"Arquivos em corpus/: {[os.path.basename(f) for f in files]}")
        with pymupdf.open(hits[0]) as pdf:
            n_pages = len(pdf)
            raw = "\n".join(page.get_text() for page in pdf)
        docs[doc_id] = {"path": hits[0], "pages": n_pages, "text": clean_text(raw)}
    return docs


docs = load_documents()
for k, d in docs.items():
    print(f"{k:24s} {d['pages']:3d} págs  {len(d['text']):7d} caracteres")

In [ ]:
# [célula 5] Chunking de tamanho fixo (com sobreposição)
def chunk_text(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks, start, n = [], 0, len(text)
    while start < n:
        end = min(start + size, n)
        if end < n:                                    # evita cortar no meio de uma palavra
            cut = text.rfind(" ", start + int(size * 0.8), end)
            if cut != -1:
                end = cut
        piece = text[start:end].strip()
        if piece:
            chunks.append(piece)
        if end >= n:
            break
        start = max(end - overlap, start + 1)
        nxt = text.find(" ", start, end)               # começa o próximo chunk no início de uma palavra
        if nxt != -1:
            start = nxt + 1
    return chunks


def build_chunks(docs):
    out = []
    for doc_id, d in docs.items():
        for i, t in enumerate(chunk_text(d["text"])):
            out.append({"chunk_id": f"{doc_id}#{i}", "doc_id": doc_id, "idx": i, "text": t})
    return out


chunks = build_chunks(docs)
print("Total de chunks:", len(chunks))
print(pd.Series([c["doc_id"] for c in chunks]).value_counts().to_string())

In [ ]:
# [célula 6] Embeddings e índice (cacheado em disco; indexação é medida à parte)
def embed_texts(texts, task_type):
    """Retorna (matriz L2-normalizada, nº de chamadas, soma das latências, retries)."""
    vecs, n_calls, lat, retries = [], 0, 0.0, 0
    for i in range(0, len(texts), EMB_BATCH):
        batch = texts[i:i + EMB_BATCH]
        res, t, r = call_with_retry(lambda b=batch: client.models.embed_content(
            model=EMB_MODEL, contents=b, config=types.EmbedContentConfig(task_type=task_type)))
        vecs += [e.values for e in res.embeddings]
        n_calls += 1; lat += t; retries += r
        if len(texts) > EMB_BATCH:
            print(f"  embeddings {min(i + EMB_BATCH, len(texts))}/{len(texts)}")
            time.sleep(EMB_SLEEP)
    arr = np.array(vecs, dtype="float32")
    arr /= np.linalg.norm(arr, axis=1, keepdims=True)
    return arr, n_calls, lat, retries


def build_or_load_index(chunks):
    sig = {"emb_model": EMB_MODEL, "chunk_size": CHUNK_SIZE, "overlap": CHUNK_OVERLAP,
           "n_chunks": len(chunks),
           "hash": hashlib.md5("".join(c["text"] for c in chunks).encode()).hexdigest()}
    meta_p, emb_p = f"{INDEX_DIR}/meta.json", f"{INDEX_DIR}/emb.npy"
    if os.path.exists(meta_p) and os.path.exists(emb_p):
        meta = json.load(open(meta_p))
        if meta.get("sig") == sig:
            print("Índice carregado do cache.")
            return np.load(emb_p), meta["stats"]
    print("Gerando embeddings dos chunks (pode levar alguns minutos no plano gratuito)...")
    emb, n_calls, lat, retries = embed_texts([c["text"] for c in chunks], "RETRIEVAL_DOCUMENT")
    tok = sum(est_tokens(c["text"]) for c in chunks)
    stats = {"chamadas_embedding": n_calls, "latencia_s": round(lat, 2), "retries": retries,
             "tokens_estimados": tok, "custo_usd": round(cost_usd(tok_emb=tok), 6), "n_chunks": len(chunks)}
    np.save(emb_p, emb)
    json.dump({"sig": sig, "stats": stats}, open(meta_p, "w"))
    json.dump(stats, open(f"{RESULTS_DIR}/baseline_indexacao.json", "w"), indent=2)
    return emb, stats


chunk_emb, index_stats = build_or_load_index(chunks)
print("Indexação:", index_stats)
print("Dimensão do embedding:", chunk_emb.shape)

In [ ]:
# [célula 7] Retrieval + geração (um pipeline RAG tradicional)
SYSTEM_PROMPT = (
    "Você é um assistente que responde perguntas sobre normas acadêmicas da UFRN "
    "(Regulamento dos Cursos de Graduação, resoluções do CONSEPE e instruções normativas da PROGRAD).\n"
    "Responda SOMENTE com base nos trechos de contexto fornecidos.\n"
    "Se os trechos não trouxerem a informação necessária, diga que não encontrou a informação nos documentos.\n"
    "Seja objetivo (no máximo 5 frases) e, quando possível, cite o documento e o artigo em que se baseou "
    "(ex.: 'art. 30, §1º do Regulamento').\n"
    "Atenção: normas diferentes podem tratar de assuntos parecidos; confira a qual documento cada trecho pertence."
)


def retrieve(question, k=TOP_K):
    t0 = time.perf_counter()
    qv, _, lat_emb, retries = embed_texts([question], "RETRIEVAL_QUERY")
    sims = chunk_emb @ qv[0]
    top = np.argsort(-sims)[:k]
    lat = time.perf_counter() - t0                     # embedding da pergunta + busca por similaridade
    hits = [{**chunks[i], "score": float(sims[i])} for i in top]
    return hits, lat, retries


def generate(question, hits, seed):
    ctx = "\n\n".join(f"[{i + 1}] Fonte: {DOC_TITLES[h['doc_id']]}\n{h['text']}" for i, h in enumerate(hits))
    prompt = f"Contexto:\n{ctx}\n\nPergunta: {question}\n\nResposta:"
    cfg = types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT, temperature=TEMPERATURE,
        max_output_tokens=MAX_OUTPUT_TOKENS, seed=seed,
        thinking_config=types.ThinkingConfig(thinking_budget=0),   # RAG tradicional: sem "thinking"
    )
    throttle_generation()
    res, lat, retries = call_with_retry(lambda: client.models.generate_content(model=GEN_MODEL, contents=prompt, config=cfg))
    _last_gen_end[0] = time.time()
    u = res.usage_metadata
    tin = getattr(u, "prompt_token_count", 0) or 0
    tout = getattr(u, "candidates_token_count", 0) or 0
    tth = getattr(u, "thoughts_token_count", 0) or 0
    return (res.text or "").strip(), lat, retries, tin, tout, tth

In [ ]:
# [célula 8] Execução do baseline (r1, r2, r3)
questions = json.load(open(QUESTIONS_PATH, encoding="utf-8"))
print("Perguntas:", len(questions))


def run_baseline(run_id):
    out_csv = f"{RESULTS_DIR}/baseline_r{run_id}.csv"
    if os.path.exists(out_csv) and not FORCE_RERUN:
        print(f"[r{run_id}] já existe ({out_csv}) — pulando. Use FORCE_RERUN=True para refazer.")
        return pd.read_csv(out_csv)
    rows, contexts = [], {}
    for q in questions:
        err = ""
        try:
            hits, lat_ret, r1 = retrieve(q["pergunta"])
            ans, lat_gen, r2, tin, tout, tth = generate(q["pergunta"], hits, seed=run_id * 1000 + q["id"])
        except Exception as e:
            err = f"{type(e).__name__}: {e}"[:300]
            hits, lat_ret, lat_gen, r1, r2, tin, tout, tth, ans = [], 0.0, 0.0, 0, 0, 0, 0, 0, ""
        tok_q = est_tokens(q["pergunta"])
        rows.append({
            "run": run_id, "id": q["id"], "tipo": q["tipo"], "pergunta": q["pergunta"],
            "gabarito": q["resposta"],
            "fontes_gabarito": "; ".join(f"{f['documento']}: {f['artigos']}" for f in q["fontes"]),
            "resposta_modelo": ans,
            "chunks_recuperados": " | ".join(f"{h['chunk_id']} ({h['score']:.3f})" for h in hits),
            "docs_recuperados": ", ".join(dict.fromkeys(h["doc_id"] for h in hits)),
            "lat_retrieval_s": round(lat_ret, 3), "lat_geracao_s": round(lat_gen, 3),
            "lat_total_s": round(lat_ret + lat_gen, 3),
            "chamadas_llm": 0 if err else 1, "chamadas_embedding": 0 if err else 1,
            "tokens_in": tin, "tokens_out": tout, "tokens_thinking": tth,
            "custo_usd": round(cost_usd(tin, tout + tth, tok_q), 6),
            "retries": r1 + r2, "erro": err,
            "acerto": "",          # <- PREENCHER MANUALMENTE: 1 = acertou, 0 = errou
        })
        contexts[q["id"]] = [{"chunk_id": h["chunk_id"], "score": h["score"], "text": h["text"]} for h in hits]
        print(f"[r{run_id}] q{q['id']:02d} ok  ret={lat_ret:.2f}s gen={lat_gen:.2f}s tokens_in={tin}" + (f"  ERRO: {err}" if err else ""))
    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False, encoding="utf-8-sig")
    json.dump(contexts, open(f"{RESULTS_DIR}/baseline_r{run_id}_contexts.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    print(f"[r{run_id}] salvo em {out_csv}")
    return df


for r in range(1, N_RUNS + 1):
    run_baseline(r)

In [ ]:
# [célula 9] Avaliação MANUAL — baixar os CSVs, marcar a coluna "acerto" (1/0) e reenviar
# 1) Baixe os CSVs (abra no Excel/Google Planilhas e preencha "acerto" com 1 ou 0).
#    Dica: "chunks_recuperados" + baseline_rN_contexts.json mostram se o erro foi de busca ou de geração.
from google.colab import files
shutil.make_archive("baseline_results", "zip", RESULTS_DIR)
files.download("baseline_results.zip")

# 2) Depois de marcar, REENVIE os CSVs preenchidos (mesmos nomes) rodando as 3 linhas abaixo:
# up = files.upload()
# for name, data in up.items(): open(f"{RESULTS_DIR}/{name}", "wb").write(data)
# print("CSVs atualizados:", list(up))

In [ ]:
# [célula 10] Tabela-resumo do baseline (acertos, latência, chamadas ao LLM, custo)
def to_score(v):
    s = str(v).strip().lower()
    if s in ("1", "1.0", "sim", "s", "ok", "x", "true"): return 1.0
    if s in ("0", "0.0", "nao", "não", "n", "false"): return 0.0
    return np.nan


def summarize():
    frames = [pd.read_csv(f, encoding="utf-8-sig") for f in sorted(glob.glob(f"{RESULTS_DIR}/baseline_r*.csv"))]
    df = pd.concat(frames, ignore_index=True)
    df["acerto_num"] = df["acerto"].map(to_score)
    n_unmarked = int(df["acerto_num"].isna().sum())
    if n_unmarked:
        print(f"ATENÇÃO: {n_unmarked} linhas sem 'acerto' marcado — acertos contam só as marcadas.\n")

    per_run = df.groupby("run").agg(
        acertos=("acerto_num", "sum"), perguntas=("id", "count"),
        lat_media_s=("lat_total_s", "mean"), chamadas_llm=("chamadas_llm", "sum"),
        custo_usd=("custo_usd", "sum"), erros=("erro", lambda s: int((s.fillna("") != "").sum())),
    ).round(4)
    per_run["acuracia_%"] = (100 * per_run["acertos"] / per_run["perguntas"]).round(1)

    print("=== Por rodada ===")
    print(per_run.to_string())
    print("\n=== Média ± desvio-padrão entre rodadas ===")
    for col in ["acertos", "acuracia_%", "lat_media_s", "chamadas_llm", "custo_usd"]:
        print(f"{col:14s} {per_run[col].mean():.3f} ± {per_run[col].std(ddof=0):.3f}")
    print(f"\nIndexação (única, fora das rodadas): {index_stats}")

    by_tipo = df.groupby("tipo")["acerto_num"].mean().mul(100).round(1)
    print("\n=== Acurácia (%) por tipo de pergunta (média das rodadas) ===")
    print(by_tipo.to_string())

    by_q = df.pivot_table(index=["id", "tipo"], columns="run", values="acerto_num")
    print("\n=== Acerto por pergunta e rodada ===")
    print(by_q.to_string())

    per_run.to_csv(f"{RESULTS_DIR}/baseline_resumo.csv")
    return per_run


summary = summarize()